In [1]:
import os
import cv2
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import accuracy_score, classification_report

In [2]:
IMG_SIZE = 64

def get_label(folder_name):
    if folder_name.lower() in ["broadleaf", "grass"]:
        return 1
    else:
        return 0


In [3]:
def load_dataset(folder_path):
    data = []
    labels = []

    for class_name in os.listdir(folder_path):
        class_path = os.path.join(folder_path, class_name)

        if not os.path.isdir(class_path):
            continue

        label = get_label(class_name)

        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)

            img = cv2.imread(img_path)
            if img is None:
                continue

            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = img / 255.0   # normalize

            data.append(img)
            labels.append(label)

    return np.array(data), np.array(labels)

In [4]:
X_train, y_train = load_dataset("train")
X_val, y_val = load_dataset("val")
X_test, y_test = load_dataset("test")

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


Train: (12275, 64, 64, 3)
Validation: (1533, 64, 64, 3)
Test: (1538, 64, 64, 3)


In [5]:
# ONE HOT ENCODING
y_train = to_categorical(y_train, 2)
y_val = to_categorical(y_val, 2)
y_test_cat = to_categorical(y_test, 2)

In [6]:
model = Sequential()

model.add(Conv2D(32, (3,3), activation='relu', input_shape=(64,64,3)))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(128, (3,3), activation='relu'))
model.add(MaxPooling2D(2,2))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(2, activation='softmax'))

c:\Users\admin\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [8]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    validation_data=(X_val, y_val),
    batch_size=16
)

Epoch 1/10
768/768 ━━━━━━━━━━━━━━━━━━━━ 60s 72ms/step - accuracy: 0.8786 - loss: 0.3004 - val_accuracy: 0.9432 - val_loss: 0.1502
Epoch 2/10
768/768 ━━━━━━━━━━━━━━━━━━━━ 58s 76ms/step - accuracy: 0.9431 - loss: 0.1536 - val_accuracy: 0.9687 - val_loss: 0.0910
Epoch 3/10
768/768 ━━━━━━━━━━━━━━━━━━━━ 91s 88ms/step - accuracy: 0.9624 - loss: 0.1022 - val_accuracy: 0.9785 - val_loss: 0.0542
Epoch 4/10
768/768 ━━━━━━━━━━━━━━━━━━━━ 71s 73ms/step - accuracy: 0.9721 - loss: 0.0744 - val_accuracy: 0.9791 - val_loss: 0.0666
Epoch 5/10
768/768 ━━━━━━━━━━━━━━━━━━━━ 45s 58ms/step - accuracy: 0.9782 - loss: 0.0638 - val_accuracy: 0.9759 - val_loss: 0.0612
Epoch 6/10
768/768 ━━━━━━━━━━━━━━━━━━━━ 49s 64ms/step - accuracy: 0.9809 - loss: 0.0582 - val_accuracy: 0.9856 - val_loss: 0.0450
Epoch 7/10
768/768 ━━━━━━━━━━━━━━━━━━━━ 52s 68ms/step - accuracy: 0.9857 - loss: 0.0417 - val_accuracy: 0.9759 - val_loss: 0.0567
Epoch 8/10
768/768 ━━━━━━━━━━━━━━━━━━━━ 47s 61ms/step - accuracy: 0.9873 - loss: 0.0380 - 

In [9]:
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

print("Test Accuracy:", accuracy_score(y_test, y_pred_classes))
print("\nClassification Report:\n", classification_report(y_test, y_pred_classes))

49/49 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step
Test Accuracy: 0.9876462938881665

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99      1065
           1       0.98      0.98      0.98       473

    accuracy                           0.99      1538
   macro avg       0.99      0.98      0.99      1538
weighted avg       0.99      0.99      0.99      1538



In [10]:
model.save("weed_model.h5")